# test-2 cosine annealing LR scheduler
* pytorch's CosineAnnealingLR implements restarts
    * SGDR: Stochastic Gradient Descent with Warm Restarts

Warmup:
* αt = (t/Tw) * αmax
    * warmup step 중 채운 비율 (t/Tw)에 비례하게 올라감

Annealing:
*  Tw ≤ t ≤ Tc, then αt = αmin + 1/2*(1 + cos(((t−Tw)/(Tc−Tw)) * π))*(αmax−αmin).

Post-annealing
* min으로 고정

In [50]:
# Implementation
import math
import torch
import torch.optim as optim
from torch.optim.optimizer import Optimizer
from torch.optim.lr_scheduler import LRScheduler

from typing_extensions import override

In [51]:
def cosine_annealing_lr(
    it: int,
    max_learning_rate: float,
    min_learning_rate: float,
    warmup_iters: int,
    cosine_cycle_iters: int,
):
    # Warmup
    if it < warmup_iters:
        return (it/warmup_iters) * max_learning_rate
    # Cosine Annealing
    elif warmup_iters<=it and it<=cosine_cycle_iters:
        lr = min_learning_rate
        lr += 0.5 * (1 + math.cos(((it-warmup_iters)*math.pi)/(cosine_cycle_iters-warmup_iters))) * (max_learning_rate-min_learning_rate)
        return lr
    # Post-annealing
    else:
        return min_learning_rate
    
class CosineAnnealingLRScheduler(LRScheduler):
    def __init__(
        self,
        optimizer: Optimizer,
        max_learning_rate: float,
        min_learning_rate: float,
        warmup_iters: int,
        cosine_cycle_iters: int,
    ):
        self.max_learning_rate=max_learning_rate
        self.min_learning_rate=min_learning_rate
        self.warmup_iters=warmup_iters
        self.cosine_cycle_iters=cosine_cycle_iters
    
        super().__init__(optimizer)
        # self._step_count = 0
    @override
    def get_lr(self) -> list[float]:
        """Compute the learning rate."""
        # Initial Steps
        # if self._isinitial:
        #     return [group["lr"] for group in self.optimizer.param_groups]
        
        step_lr = cosine_annealing_lr(
            it=self._step_count,
            max_learning_rate=self.max_learning_rate,
            min_learning_rate=self.min_learning_rate,
            warmup_iters=self.warmup_iters,
            cosine_cycle_iters=self.cosine_cycle_iters,
        )
        return [
            step_lr for group in self.optimizer.param_groups
        ]

In [52]:
# Random Optimizer
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))


In [53]:
max_steps = 100

lr = 1.0
warmup_ratio = 0.1
min_learning_rate = 1e-2

warmup_iters = int(max_steps*warmup_ratio)

# testing - about 3 steps of warmup ratio
cosine_cycle_iters = int(max_steps * warmup_ratio * 3)

In [54]:
# Usage
optimizer = optim.SGD([weights], lr=1)

In [55]:
# super().__init__(optimizer) calls `_initial_step` -> sets _step_count
scheduler = CosineAnnealingLRScheduler(
    optimizer,
    max_learning_rate = lr,
    min_learning_rate = min_learning_rate,
    warmup_iters = warmup_iters,
    cosine_cycle_iters=cosine_cycle_iters
)

In [56]:
print(scheduler._step_count)

1


In [57]:
scheduler.get_lr()

[0.1]

In [58]:
optimizer.zero_grad() # Reset the gradients for all learnable parameters.
loss = (weights**2).mean() # Compute a scalar loss value.
loss.backward() # Run backward pass, which computes gradients.

In [59]:
optimizer.step() # Run optimizer step

In [60]:
print(scheduler._step_count)

1


In [61]:
scheduler.step()

In [62]:
print(scheduler._step_count)

2


In [63]:
scheduler.get_lr()

[0.2]